# SmartDroneInspection: MBDD2025 UAV Building Defect Detection (YOLO11s)

Notebook huấn luyện mô hình **YOLO11s** trên bộ dữ liệu **MBDD2025** (14.471 ảnh UAV chụp bằng Drone 4K).

### 5 Lớp khuyết tật mục tiêu:
1. `crack` (Vết nứt bê tông/kết cấu)
2. `leakage` (Thấm dột, ẩm ướt)
3. `abscission` (Bong tróc, vỡ lở bề mặt)
4. `corrosion` (Rỉ sét, ăn mòn kết cấu kim loại/thép)
5. `bulge` (Phồng rộp bề mặt vữa/kết cấu)

### Cài đặt môi trường Kaggle:
- **Accelerator:** GPU T4 x 2 hoặc GPU P100
- **Internet:** Turn ON (Cài đặt bên phải menu)
- **Dataset:** Add dataset `mennamahmoudd/mbdd2025-building-defects` hoặc `asadibehnam/mbdd2025-uav-building-defect-detection`

In [ ]:
# [Cell 1] Cài đặt thư viện & kiểm tra GPU
!pip install -q ultralytics opencv-python pyyaml scikit-learn

import torch
import ultralytics

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu bên phải chọn Accelerator -> GPU T4 x 2.")

In [ ]:
# [Cell 2] Chuẩn hóa Dataset: Chuyển đổi Pascal VOC XML -> YOLO format & Phân chia Train/Val/Test
import os
import glob
from pathlib import Path
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split

# 1. Tìm đường dẫn dữ liệu MBDD2025 trong /kaggle/input/
candidates = glob.glob("/kaggle/input/**/JPEGImages", recursive=True)
if not candidates:
    candidates = glob.glob("/kaggle/input/**/images", recursive=True)

if not candidates:
    raise FileNotFoundError(
        "Chưa tìm thấy thư mục ảnh! Hãy bấm nút '+ Add Data' ở góc phải notebook và thêm 'mbdd2025-building-defects'."
    )

IMG_DIR = Path(candidates[0])
DATASET_ROOT = IMG_DIR.parent
ANNOT_DIR = DATASET_ROOT / "Annotations"

print(f"Thư mục ảnh: {IMG_DIR}")
print(f"Thư mục nhãn XML: {ANNOT_DIR}")

# 2. Khai báo 5 lớp chuẩn hóa
CLASSES = ['crack', 'leakage', 'abscission', 'corrosion', 'bulge']
CLASS_MAP = {c: i for i, c in enumerate(CLASSES)}

OUTPUT_DIR = Path("/kaggle/working/dataset")
for split in ['train', 'val', 'test']:
    os.makedirs(OUTPUT_DIR / f"images/{split}", exist_ok=True)
    os.makedirs(OUTPUT_DIR / f"labels/{split}", exist_ok=True)

# Lấy danh sách ảnh
valid_exts = ('.jpg', '.jpeg', '.png')
all_imgs = [p for p in IMG_DIR.glob("*.*") if p.suffix.lower() in valid_exts]
print(f"Tổng số ảnh tìm thấy: {len(all_imgs)}")

# Chia dữ liệu 70% Train - 20% Val - 10% Test (chuẩn bài báo MBDD2025)
train_imgs, temp_imgs = train_test_split(all_imgs, test_size=0.30, random_state=42)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.3333, random_state=42)

def convert_and_link(images, split_name):
    valid_count = 0
    box_count = 0
    for img_p in images:
        xml_p = ANNOT_DIR / f"{img_p.stem}.xml"
        if not xml_p.exists():
            continue

        try:
            tree = ET.parse(xml_p)
            root = tree.getroot()
            size = root.find('size')
            w = float(size.findtext('width', '1280'))
            h = float(size.findtext('height', '720'))
            if w <= 0 or h <= 0:
                continue

            yolo_boxes = []
            for obj in root.findall('object'):
                name = obj.findtext('name', '').strip().lower()
                if name not in CLASS_MAP:
                    continue
                cls_id = CLASS_MAP[name]
                bnd = obj.find('bndbox')
                if bnd is None:
                    continue

                xmin = max(0.0, float(bnd.findtext('xmin')))
                ymin = max(0.0, float(bnd.findtext('ymin')))
                xmax = min(w, float(bnd.findtext('xmax')))
                ymax = min(h, float(bnd.findtext('ymax')))

                if xmax <= xmin or ymax <= ymin:
                    continue

                # Tọa độ chuẩn hóa [x_center, y_center, width, height]
                x_c = ((xmin + xmax) / 2.0) / w
                y_c = ((ymin + ymax) / 2.0) / h
                b_w = (xmax - xmin) / w
                b_h = (ymax - ymin) / h
                yolo_boxes.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {b_w:.6f} {b_h:.6f}")

            if yolo_boxes:
                # Symlink ảnh để không tốn ổ đĩa và chạy siêu tốc
                target_img = OUTPUT_DIR / f"images/{split_name}" / img_p.name
                if not target_img.exists():
                    os.symlink(img_p, target_img)

                target_lbl = OUTPUT_DIR / f"labels/{split_name}" / f"{img_p.stem}.txt"
                target_lbl.write_text("\n".join(yolo_boxes))
                valid_count += 1
                box_count += len(yolo_boxes)
        except Exception:
            pass

    print(f"[{split_name.upper()}] {valid_count} ảnh hợp lệ, {box_count} bounding boxes.")

convert_and_link(train_imgs, 'train')
convert_and_link(val_imgs, 'val')
convert_and_link(test_imgs, 'test')

# 3. Ghi file cấu hình data.yaml
data_yaml_content = f"""
path: {OUTPUT_DIR.resolve()}
train: images/train
val: images/val
test: images/test

names:
  0: crack
  1: leakage
  2: abscission
  3: corrosion
  4: bulge
"""
yaml_path = Path("/kaggle/working/mbdd2025.yaml")
yaml_path.write_text(data_yaml_content.strip())
print("\nĐã tạo cấu hình YOLO data.yaml tại:", yaml_path)

In [ ]:
# [Cell 3] Huấn luyện YOLO11s (MBDD2025 Benchmark Hyperparameters)
from ultralytics import YOLO

# Khởi tạo model YOLO11s pretrained
model = YOLO("yolo11s.pt")

train_args = dict(
    data="/kaggle/working/mbdd2025.yaml",
    epochs=100,             # 100 epoch theo benchmark Nature Scientific Data
    imgsz=640,              # Chuẩn kích thước ảnh tối ưu tốc độ/độ chính xác
    batch=16,               # Batch size phù hợp GPU Kaggle T4/P100 (16GB VRAM)
    patience=20,            # Tự dừng sớm nếu 20 epoch không tăng mAP
    device=0,
    workers=4,
    optimizer="AdamW",      # Optimizer thích ứng gradient nhanh
    lr0=0.002,
    cos_lr=True,            # Cosine Learning Rate Schedule giúp hội tụ mượt
    scale=0.5,              # Augmentation: Thu phóng 50%-150% (giả lập khoảng cách drone 5m-25m)
    fliplr=0.5,             # Augmentation: Lật ngang ảnh
    mosaic=1.0,             # Ghép 4 góc cảnh giúp model nhận diện đa khuyết tật
    close_mosaic=10,        # Tắt mosaic 10 epoch cuối để tinh chỉnh bounding box
    project="/kaggle/working/runs",
    name="smartdrone_mbdd_yolo11s",
    save_period=5           # Cứ 5 epoch tự động lưu 1 checkpoint
)

print("Bắt đầu huấn luyện YOLO11s...")
results = model.train(**train_args)

In [ ]:
# [Cell 4] Đánh giá chi tiết trên tập Test & Xuất kết quả báo cáo
import os
import pandas as pd

best_weights = "/kaggle/working/runs/smartdrone_mbdd_yolo11s/weights/best.pt"
if os.path.exists(best_weights):
    trained_model = YOLO(best_weights)
    metrics = trained_model.val(data="/kaggle/working/mbdd2025.yaml", split="test")
    
    print("\n" + "="*50)
    print("KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST:")
    print(f"- Overall Precision: {metrics.box.mp:.4f}")
    print(f"- Overall Recall:    {metrics.box.mr:.4f}")
    print(f"- mAP@0.5:           {metrics.box.map50:.4f}")
    print(f"- mAP@0.5:0.95:      {metrics.box.map:.4f}")
    print("="*50)
    
    # Chi tiết từng class
    class_names = ['crack', 'leakage', 'abscission', 'corrosion', 'bulge']
    for i, name in enumerate(class_names):
        if i < len(metrics.box.maps):
            print(f"{name:12s} | mAP@0.5: {metrics.box.ap50[i]:.4f} | mAP@0.5:0.95: {metrics.box.maps[i]:.4f}")
else:
    print("Không tìm thấy best.pt!")

In [ ]:
# [Cell 5] Đóng gói Artifacts (best.pt, best.onnx, biểu đồ) ra thư mục /kaggle/working/export/
import shutil

EXPORT_DIR = Path("/kaggle/working/export")
os.makedirs(EXPORT_DIR, exist_ok=True)

if os.path.exists(best_weights):
    trained_model = YOLO(best_weights)
    # 1. Export ONNX (Chuẩn hóa để nhúng drone hoặc deploy backend)
    trained_model.export(format="onnx", imgsz=640, simplify=True)
    
    # 2. Copy weights và ONNX
    shutil.copy(best_weights, EXPORT_DIR / "best.pt")
    onnx_path = best_weights.replace(".pt", ".onnx")
    if os.path.exists(onnx_path):
        shutil.copy(onnx_path, EXPORT_DIR / "best.onnx")
        
    # 3. Copy các biểu đồ huấn luyện để đưa vào báo cáo đồ án
    run_dir = Path("/kaggle/working/runs/smartdrone_mbdd_yolo11s")
    for plot_file in ["results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"]:
        if (run_dir / plot_file).exists():
            shutil.copy(run_dir / plot_file, EXPORT_DIR / plot_file)
            
    print("\nĐÃ ĐÓNG GÓI THÀNH CÔNG VÀO /kaggle/working/export/")
    print("Danh sách file trong export:", os.listdir(EXPORT_DIR))
    print("\n>>> Bạn có thể tải toàn bộ file này từ tab 'Output' bên phải giao diện Kaggle!")

In [ ]:
# [Cell 6] Tạo script FastAPI Service chuẩn kết nối Backend Spring Boot
service_code = '''import io
from PIL import Image
from fastapi import FastAPI, UploadFile, File
from ultralytics import YOLO

app = FastAPI(title="SmartDroneInspection AI Inference Service")
model = YOLO("best.pt")

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    image_bytes = await file.read()
    image = Image.open(io.BytesIO(image_bytes))
    
    # Inference với confidence threshold 0.25
    results = model.predict(source=image, conf=0.25)
    detections = []
    
    for r in results:
        for box in r.boxes:
            coords = box.xyxyn[0].tolist()  # [xMin, yMin, xMax, yMax] normalized
            label = model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            
            detections.append({
                "modelName": "yolo11s",
                "modelVersion": "mbdd2025-v1",
                "predictedLabel": label,
                "confidence": round(conf, 4),
                "boundingBox": f'{{"xMin": {coords[0]:.4f}, "yMin": {coords[1]:.4f}, "xMax": {coords[2]:.4f}, "yMax": {coords[3]:.4f}}}'
            })
            
    return detections

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

Path("/kaggle/working/export/app.py").write_text(service_code.strip())
print("Đã tạo app.py tương thích 100% với Spring Boot backend YoloInferenceClient!")